In [33]:
from typing import Dict, TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain.vectorstores import FAISS

from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=gemini_api_key)
os.environ["GOOGLE_API_KEY"] = gemini_api_key

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

class GraphState(TypedDict):
    question: Optional[str] = None
    classification: Optional[str] = None
    response: Optional[str] = None
    length: Optional[int] = None
    greeting: Optional[str] = None

workflow = StateGraph(GraphState)


def retriever_qa_creation():
       pass
rag_chain = retriever_qa_creation()

from langchain_core.messages import HumanMessage

def classify(question):
    return llm([HumanMessage(content=f"classify intent of given input as greeting or not_greeting. Output just the class. Input: {question}")])

def classify_input_node(state):
    question = state.get('question', '').strip()

    classification = classify(question) 
    print("___________________________________________")
    print(classification)
    return {"classification": classification.content}

def handle_greeting_node(state):
    return {"greeting": "Hello! How can I help you today?"}

def handle_RAG(state):
    question = state.get('question', '').strip()
    prompt = question
    if state.get("length",0)<30:
         search_result = rag_chain.run(prompt)
    else:
         search_result = rag_chain.run(prompt+'. Return total count only.')

    return {"response": search_result,"length":len(search_result)}


def bye(state):
    return{"greeting":"The graph has finished"}

workflow.add_node("classify_input", classify_input_node)
workflow.add_node("handle_greeting", handle_greeting_node)
workflow.add_node("handle_RAG", handle_RAG)
workflow.add_node("bye", bye)

workflow.set_entry_point("classify_input")
workflow.add_edge('handle_greeting', END)
workflow.add_edge('bye', END)


def decide_next_node(state):
    return "handle_greeting" if state.get('classification') == "greeting" else "handle_RAG"

def check_RAG_length(state):
    print(state.get("length",0))
    if state.get("length",0)>30 :
        return "handle_RAG"
    else:
        return "bye"

workflow.add_conditional_edges(
    "classify_input",
    decide_next_node,
    {
        "handle_greeting": "handle_greeting",
        "handle_RAG": "handle_RAG"
    }
)

workflow.add_conditional_edges(
    "handle_RAG",
    check_RAG_length,
    {
        "bye": "bye",
        "handle_RAG": "handle_RAG"
    }
)

app = workflow.compile()
app.invoke({'question':'Hello bot','length':0})

C:\Users\Parthiban\AppData\Local\Temp\ipykernel_20368\3330063374.py:38: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return llm([HumanMessage(content=f"classify intent of given input as greeting or not_greeting. Output just the class. Input: {question}")])


___________________________________________
content='greeting' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run-f98f06bf-c7e7-4378-9114-3fc47309b0cf-0' usage_metadata={'input_tokens': 21, 'output_tokens': 2, 'total_tokens': 23, 'input_token_details': {'cache_read': 0}}


{'question': 'Hello bot',
 'classification': 'greeting',
 'length': 0,
 'greeting': 'Hello! How can I help you today?'}

In [27]:
def llm_response():
    pass


def web_search():
    pass

def db_query():
    pass


def summarize_results():
    pass

def generate_response():
    pass


def rate_limit():
    pass


from langgraph.graph import StateGraph,START,END
from typing import TypedDict
class GraphState(TypedDict):
    rate_limiter:int
    try_web_Search:bool
    try_db_query:bool


workflow = StateGraph(GraphState)
workflow.add_node("llm_response",llm_response)
workflow.add_node("web_search",web_search)
workflow.add_node("db_query",db_query)
workflow.add_node("summarize_results",summarize_results)
workflow.add_node("rate_limit",rate_limit)


workflow.set_entry_point("llm_response")
workflow.add_edge("web_search","summarize_results")
workflow.add_edge("db_query","summarize_results")
workflow.add_edge("summarize_results","llm_response")

workflow.add_conditional_edges(
    "llm_response",
    lambda state:"web_search" if state.get("try_web_Search") else "try_db_query",
    {
        "web_search":"web_search",
        "db_query":"db_query",
    
    }
)



graph = workflow.compile()



In [28]:
from IPython.display import Image, display
# Generate just the Mermaid syntax
print(graph.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	llm_response(llm_response)
	web_search(web_search)
	db_query(db_query)
	summarize_results(summarize_results)
	rate_limit([rate_limit]):::last
	__start__ --> llm_response;
	db_query --> summarize_results;
	summarize_results --> llm_response;
	web_search --> summarize_results;
	llm_response -.-> web_search;
	llm_response -.-> db_query;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

